# A1 post-hoc analyses and cost benchmark (Kaggle, about 1 h of one T4)
Runs the GPU-only parts of the protocol on the trained models, with no training: **cost benchmark** (`cost_bench`), **robustness** (6 corruptions x 3 severities, 8 core models), **Grad-CAM** (12 pre-selected images) and the **highest-loss training images**. The weights come from the Hugging Face store.

Settings (right panel): Accelerator **GPU T4 x2** (only GPU 0 is used), Internet **On**, *Add Input* → the private Food-101 dataset, *Add-ons → Secrets* → `HF_TOKEN`.
**Run the cost benchmark first and alone** (cell 4): it measures speed, so nothing else may use the GPU at that time.

Outputs go to `/kaggle/working/posthoc_out/`; download that folder and commit: `results/*.json` → `A1/results/`, `figures/*` → `A1/figures/`.

In [ ]:
# 1) settings
HF_REPO = "dl4dl/a1-store"            # TODO: same as in the worker notebook
REPO_URL = "https://github.com/broistg/deep-learning-CO5085.git"
BRANCH = "main"

In [ ]:
# 2) set up (torch is preinstalled and not touched)
import os, subprocess, shutil
REPO = "/kaggle/working/repo"
def sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
if not os.path.isdir(REPO + "/.git"):
    sh(f"git clone -b {BRANCH} {REPO_URL} {REPO}")
else:
    sh(f"cd {REPO} && git pull --ff-only")
os.chdir(REPO)
sh("pip install -q -r requirements/a1.txt")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
OUT = "/kaggle/working/posthoc_out"
os.environ.update(A1_STORE=f"hf:{HF_REPO}", A1_LOCAL="/kaggle/temp/a1_local", PYTHONPATH=f"{REPO}:{REPO}/A1", CUDA_VISIBLE_DEVICES="0")
os.makedirs(OUT + "/results", exist_ok=True); os.makedirs(OUT + "/figures", exist_ok=True)
sh("nvidia-smi -L; git log -1 --format='commit %h %s'; ls A1/results | head -3")

In [ ]:
# 3) the 8 core models' weights from the store (about 0.8 GB) and the dataset copy to local disk (about 5 min)
sh("python -m a1lib.collect fetch-core --out /kaggle/temp/a1_artifacts")
sh("python -m a1lib.worker prepare")

In [ ]:
# 4) COST BENCHMARK (alone on the GPU, about 10 min): GPU-only s/epoch per architecture and mode + inference cost
sh(f"python -m a1lib.cost_bench --out {OUT}/results")

In [ ]:
# 5) robustness: 8 models x (clean + 18 corrupted) on the fixed 5,050-image subset (about 15-20 min)
sh(f"python -m a1lib.posthoc robustness --results A1/results --artifacts /kaggle/temp/a1_artifacts --workers 4")
shutil.copy("A1/results/posthoc_robustness.json", OUT + "/results/")

In [ ]:
# 6) Grad-CAM on the 12 images chosen by the fixed rule (every 9th class), and the highest-loss training images (about 10 min)
sh(f"python -m a1lib.posthoc gradcam --results A1/results --artifacts /kaggle/temp/a1_artifacts --out {OUT}")
sh(f"python -m a1lib.posthoc toploss --results A1/results --artifacts /kaggle/temp/a1_artifacts --out {OUT}")
for f in ("posthoc_gradcam.json", "posthoc_toploss.json"):
    shutil.copy("A1/results/" + f, OUT + "/results/")

In [ ]:
# 7) pack the small outputs for download
shutil.make_archive("/kaggle/working/a1_posthoc", "zip", OUT)
sh("ls -la /kaggle/working/a1_posthoc.zip; unzip -l /kaggle/working/a1_posthoc.zip")